# Experiment: U-Net And Swin-Unet Comparison On ETIS

这一册完整实现 U-Net 和 Swin-Unet，并与 `01` 中保存的 EMCAD baseline 结果做统一口径比较。


## 加载基础工具

这里只使用环境与结果保存工具，所有 ETIS 数据读取和模型定义都写在本 notebook 内。


In [ ]:
from pathlib import Path
import json
import os
import random
import sys

from scripts.project_utils import (
    ARTIFACTS,
    DATA_ROOT,
    ETIS_ROOT,
    PVT_PRETRAINED_ROOT,
    ensure_project_dirs,
    get_default_project_config,
    load_project_config,
    print_env_summary,
    save_json,
    set_seed,
    try_import_torch,
)

ensure_project_dirs()
set_seed()
torch = try_import_torch()
ENV_SUMMARY = print_env_summary(torch)
ROOT = Path.cwd().resolve()


## 读取共享实验配置

这里直接复用 `00` 生成的统一配置，保证和 `01` 的 ETIS 路径、训练参数与固定可视化样本保持一致。


In [ ]:
PROJECT_CONFIG = load_project_config()
PROJECT_CONFIG


## 复用 ETIS 数据与评估工具

这里直接复用 `01` 的 ETIS 数据流与 Dice 评估逻辑，确保对照实验和 EMCAD baseline 完全同口径。


In [ ]:
import json
from pathlib import Path

baseline_nb = json.loads(Path("01_emcad_full_training.ipynb").read_text(encoding="utf-8"))
for idx in [2, 4, 6, 8]:
    exec("".join(baseline_nb["cells"][idx]["source"]), globals())


## 定义 U-Net

这一段采用接近官方 `Pytorch-UNet` 的经典编码器 - 解码器结构，保留四次下采样、四次上采样和 skip connection。


In [ ]:
assert torch is not None, "需要先安装 PyTorch 才能运行本 notebook。"

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        mid_channels = mid_channels or out_channels
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels))

    def forward(self, x):
        return self.block(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diff_y = x2.size(2) - x1.size(2)
        diff_x = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2, diff_y // 2, diff_y - diff_y // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

class UNetBaseline(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, base_channels=32, bilinear=True):
        super().__init__()
        factor = 2 if bilinear else 1
        c1, c2, c3, c4, c5 = (
            base_channels,
            base_channels * 2,
            base_channels * 4,
            base_channels * 8,
            base_channels * 16,
        )
        self.inc = DoubleConv(n_channels, c1)
        self.down1 = Down(c1, c2)
        self.down2 = Down(c2, c3)
        self.down3 = Down(c3, c4)
        self.down4 = Down(c4, c5 // factor)
        self.up1 = Up(c5, c4 // factor, bilinear=bilinear)
        self.up2 = Up(c4, c3 // factor, bilinear=bilinear)
        self.up3 = Up(c3, c2 // factor, bilinear=bilinear)
        self.up4 = Up(c2, c1, bilinear=bilinear)
        self.head = OutConv(c1, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.head(x)


## 定义 Swin-Unet

这一段参考官方 `Swin-Unet` 的核心思路，保留 patch embedding、分层 Swin Transformer block、patch merging、patch expand 和 U 形跳连解码结构，而不是只用简单卷积替代。


In [ ]:
assert torch is not None, "需要先安装 PyTorch 才能运行本 notebook。"

class SwinMLP(nn.Module):
    def __init__(self, dim, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        hidden_dim = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        return self.drop(x)

def window_partition(x, window_size):
    b, h, w, c = x.shape
    x = x.view(b, h // window_size, window_size, w // window_size, window_size, c)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    return windows.view(-1, window_size * window_size, c)

def window_reverse(windows, window_size, h, w):
    b = int(windows.shape[0] / ((h // window_size) * (w // window_size)))
    x = windows.view(b, h // window_size, w // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    return x.view(b, h, w, -1)

class WindowAttention(nn.Module):
    def __init__(self, dim, window_size=7, num_heads=3, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        relative_size = (2 * window_size - 1) * (2 * window_size - 1)
        self.relative_position_bias_table = nn.Parameter(torch.zeros(relative_size, num_heads))
        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)
        coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing="ij"))
        coords_flat = coords.flatten(1)
        relative_coords = coords_flat[:, :, None] - coords_flat[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer("relative_position_index", relative_position_index)
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        b_, n, c = x.shape
        qkv = self.qkv(x).reshape(b_, n, 3, self.num_heads, c // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)]
        relative_position_bias = relative_position_bias.view(
            self.window_size * self.window_size,
            self.window_size * self.window_size,
            -1,
        )
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)
        if mask is not None:
            num_windows = mask.shape[0]
            attn = attn.view(b_ // num_windows, num_windows, self.num_heads, n, n)
            attn = attn + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, n, n)
        attn = self.softmax(attn)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(b_, n, c)
        x = self.proj(x)
        return self.proj_drop(x)

class SwinTransformerBlock(nn.Module):
    def __init__(
        self,
        dim,
        input_resolution,
        num_heads,
        window_size=7,
        shift_size=0,
        mlp_ratio=4.0,
        qkv_bias=True,
        drop=0.0,
        attn_drop=0.0,
    ):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.window_size = window_size
        self.shift_size = shift_size if min(input_resolution) > window_size else 0
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(
            dim=dim,
            window_size=window_size,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
        )
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = SwinMLP(dim=dim, mlp_ratio=mlp_ratio, drop=drop)
        if self.shift_size > 0:
            self.register_buffer("attn_mask", self._build_mask(input_resolution))
        else:
            self.attn_mask = None

    def _build_mask(self, input_resolution):
        h, w = input_resolution
        padded_h = int(math.ceil(h / self.window_size)) * self.window_size
        padded_w = int(math.ceil(w / self.window_size)) * self.window_size
        mask = torch.zeros((1, padded_h, padded_w, 1))
        h_slices = (
            slice(0, -self.window_size),
            slice(-self.window_size, -self.shift_size),
            slice(-self.shift_size, None),
        )
        w_slices = (
            slice(0, -self.window_size),
            slice(-self.window_size, -self.shift_size),
            slice(-self.shift_size, None),
        )
        count = 0
        for h_slice in h_slices:
            for w_slice in w_slices:
                mask[:, h_slice, w_slice, :] = count
                count += 1
        mask_windows = window_partition(mask, self.window_size).view(-1, self.window_size * self.window_size)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        return attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, 0.0)

    def forward(self, x):
        h, w = self.input_resolution
        b, l, c = x.shape
        assert l == h * w, "token length and resolution do not match"
        shortcut = x
        x = self.norm1(x).view(b, h, w, c)
        pad_h = (self.window_size - h % self.window_size) % self.window_size
        pad_w = (self.window_size - w % self.window_size) % self.window_size
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x.permute(0, 3, 1, 2), (0, pad_w, 0, pad_h)).permute(0, 2, 3, 1)
        padded_h, padded_w = x.shape[1], x.shape[2]
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x
        x_windows = window_partition(shifted_x, self.window_size)
        attn_windows = self.attn(x_windows, mask=self.attn_mask)
        shifted_x = window_reverse(attn_windows, self.window_size, padded_h, padded_w)
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x
        x = x[:, :h, :w, :].contiguous().view(b, h * w, c)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x

class PatchEmbed(nn.Module):
    def __init__(self, img_size=352, patch_size=4, in_channels=3, embed_dim=48):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        h, w = x.shape[2], x.shape[3]
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x, h, w

class PatchMerging(nn.Module):
    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.norm = nn.LayerNorm(dim * 4)
        self.reduction = nn.Linear(dim * 4, dim * 2, bias=False)

    def forward(self, x):
        h, w = self.input_resolution
        b, l, c = x.shape
        assert l == h * w, "token length and resolution do not match"
        x = x.view(b, h, w, c)
        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]
        x = torch.cat([x0, x1, x2, x3], dim=-1)
        x = x.view(b, -1, 4 * c)
        x = self.norm(x)
        return self.reduction(x)

class PatchExpand(nn.Module):
    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.expand = nn.Linear(dim, dim * 2, bias=False)
        self.norm = nn.LayerNorm(dim // 2)

    def forward(self, x):
        h, w = self.input_resolution
        b, l, c = x.shape
        assert l == h * w, "token length and resolution do not match"
        x = self.expand(x)
        x = x.view(b, h, w, 2, 2, c // 2)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(b, h * 2 * w * 2, c // 2)
        return self.norm(x)

class FinalPatchExpandX4(nn.Module):
    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.expand = nn.Linear(dim, dim * 16, bias=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        h, w = self.input_resolution
        b, l, c = x.shape
        assert l == h * w, "token length and resolution do not match"
        x = self.expand(x)
        x = x.view(b, h, w, 4, 4, c)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(b, h * 4, w * 4, c)
        x = x.view(b, -1, c)
        return self.norm(x)

class BasicLayer(nn.Module):
    def __init__(self, dim, input_resolution, depth, num_heads, window_size=7, mlp_ratio=4.0):
        super().__init__()
        self.blocks = nn.ModuleList(
            [
                SwinTransformerBlock(
                    dim=dim,
                    input_resolution=input_resolution,
                    num_heads=num_heads,
                    window_size=window_size,
                    shift_size=0 if idx % 2 == 0 else window_size // 2,
                    mlp_ratio=mlp_ratio,
                )
                for idx in range(depth)
            ]
        )
        self.input_resolution = input_resolution

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x

class SkipFusion(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x, skip):
        x = torch.cat([x, skip], dim=-1)
        x = self.norm(x)
        return self.proj(x)

class SwinUNetBaseline(nn.Module):
    def __init__(
        self,
        img_size=352,
        patch_size=4,
        in_channels=3,
        num_classes=1,
        embed_dim=48,
        depths=(2, 2, 2, 2),
        num_heads=(3, 6, 12, 24),
        window_size=7,
    ):
        super().__init__()
        assert img_size % patch_size == 0, "img_size must be divisible by patch_size"
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size, in_channels=in_channels, embed_dim=embed_dim)
        resolution = img_size // patch_size
        dims = [embed_dim, embed_dim * 2, embed_dim * 4, embed_dim * 8]
        self.encoder_layers = nn.ModuleList(
            [
                BasicLayer(dims[0], (resolution, resolution), depths[0], num_heads[0], window_size=window_size),
                BasicLayer(dims[1], (resolution // 2, resolution // 2), depths[1], num_heads[1], window_size=window_size),
                BasicLayer(dims[2], (resolution // 4, resolution // 4), depths[2], num_heads[2], window_size=window_size),
                BasicLayer(dims[3], (resolution // 8, resolution // 8), depths[3], num_heads[3], window_size=window_size),
            ]
        )
        self.downsamples = nn.ModuleList(
            [
                PatchMerging((resolution, resolution), dims[0]),
                PatchMerging((resolution // 2, resolution // 2), dims[1]),
                PatchMerging((resolution // 4, resolution // 4), dims[2]),
            ]
        )
        self.up3 = PatchExpand((resolution // 8, resolution // 8), dims[3])
        self.up2 = PatchExpand((resolution // 4, resolution // 4), dims[2])
        self.up1 = PatchExpand((resolution // 2, resolution // 2), dims[1])
        self.fuse3 = SkipFusion(dims[2] * 2, dims[2])
        self.fuse2 = SkipFusion(dims[1] * 2, dims[1])
        self.fuse1 = SkipFusion(dims[0] * 2, dims[0])
        self.decoder3 = BasicLayer(dims[2], (resolution // 4, resolution // 4), 2, num_heads[2], window_size=window_size)
        self.decoder2 = BasicLayer(dims[1], (resolution // 2, resolution // 2), 2, num_heads[1], window_size=window_size)
        self.decoder1 = BasicLayer(dims[0], (resolution, resolution), 2, num_heads[0], window_size=window_size)
        self.final_expand = FinalPatchExpandX4((resolution, resolution), dims[0])
        self.head = nn.Conv2d(embed_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x, h, w = self.patch_embed(x)
        skip1 = self.encoder_layers[0](x)
        x = self.downsamples[0](skip1)
        skip2 = self.encoder_layers[1](x)
        x = self.downsamples[1](skip2)
        skip3 = self.encoder_layers[2](x)
        x = self.downsamples[2](skip3)
        x = self.encoder_layers[3](x)

        x = self.up3(x)
        x = self.fuse3(x, skip3)
        x = self.decoder3(x)
        x = self.up2(x)
        x = self.fuse2(x, skip2)
        x = self.decoder2(x)
        x = self.up1(x)
        x = self.fuse1(x, skip1)
        x = self.decoder1(x)
        x = self.final_expand(x)
        b = x.shape[0]
        x = x.view(b, h * 4, w * 4, self.head.in_channels).permute(0, 3, 1, 2).contiguous()
        return self.head(x)


## 读取 EMCAD baseline 结果

本 notebook 不再重复定义 EMCAD，只读取 `01` 里保存的基准实验结果。


In [ ]:
emcad_result_path = ARTIFACTS / "records" / "emcad_etis_results.json"
emcad_reference = json.loads(emcad_result_path.read_text(encoding="utf-8")) if emcad_result_path.exists() else {}
emcad_reference.get("test_summary", {})


## 构建 ETIS dataloader

这里继续沿用统一的 `train / val / test = 156 / 20 / 20`。


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = PROJECT_CONFIG["train"]
train_loader, val_loader, test_loader = build_dataloaders(cfg)
{
    "device": device,
    "train_count": len(train_loader.dataset),
    "val_count": len(val_loader.dataset),
    "test_count": len(test_loader.dataset),
    "visual_sample": default_visual_sample(),
}


## 训练 U-Net 和 Swin-Unet 并在 ETIS 上评估

这里为两个模型执行统一流程，确保与 EMCAD baseline 的比较具有同一数据口径。


In [ ]:
comparison_rows = []
model_specs = {
    "U-Net": UNetBaseline(base_channels=32),
    "Swin-Unet": SwinUNetBaseline(embed_dim=48),
}

for model_name, model in model_specs.items():
    model = model.to(device)
    history, best_val_dice = train_model(model, train_loader, val_loader, cfg, device=device)
    val_summary, val_rows = evaluate_loader(model, val_loader, device=device)
    test_summary, test_rows = evaluate_loader(model, test_loader, device=device)
    history_path = ARTIFACTS / "figures" / f"{model_name.lower().replace('-', '_')}_history.png"
    visual_path = ARTIFACTS / "figures" / f"{model_name.lower().replace('-', '_')}_visual_sample.png"
    saved_history_path = save_history_figure(history, history_path)
    saved_visual_path = export_visualization(model, default_visual_sample(), cfg["image_size"], visual_path, device=device)
    checkpoint_path = ARTIFACTS / "checkpoints" / f"{model_name.lower().replace('-', '_')}_etis_best.pth"
    torch.save(model.state_dict(), checkpoint_path)
    comparison_rows.append(
        {
            "model": model_name,
            "best_val_dice": best_val_dice,
            "val_summary": val_summary,
            "test_summary": test_summary,
            "val_rows": val_rows,
            "test_rows": test_rows,
            "history_figure_path": saved_history_path,
            "visual_path": saved_visual_path,
            "checkpoint_path": str(checkpoint_path),
        }
    )

if emcad_reference:
    comparison_rows.append(
        {
            "model": "EMCAD",
            "best_val_dice": emcad_reference["best_val_dice"],
            "val_summary": emcad_reference["val_summary"],
            "test_summary": emcad_reference["test_summary"],
            "val_rows": emcad_reference["val_rows"],
            "test_rows": emcad_reference["test_rows"],
            "history_figure_path": emcad_reference["history_figure_path"],
            "visual_path": emcad_reference["visual_path"],
            "checkpoint_path": emcad_reference["checkpoint_path"],
        }
    )

comparison_rows


## 保存 ETIS 对照结果

输出表中会包含每个模型对同一个测试样本导出的分割图路径，便于后续肉眼直接比较。


In [ ]:
save_json(
    {
        "dataset": "ETIS",
        "device": device,
        "visual_sample": default_visual_sample(),
        "rows": comparison_rows,
    },
    ARTIFACTS / "records" / "baseline_comparison_etis.json",
)
